In [0]:
spark.sql("""
    SELECT
        fuel_type_id,
        fuel_type,
        sector_id,
        sector,
        COUNT(*) AS record_count,
        ROUND(SUM(generation_mwh), 2) AS total_generation_mwh
    FROM virginia_energy_silver
    GROUP BY
        fuel_type_id,
        fuel_type,
        sector_id,
        sector
    ORDER BY
        fuel_type_id,
        sector_id
""").show(100, truncate=False)

+------------+----------------------------------------+---------+-----------------------------+------------+--------------------+
|fuel_type_id|fuel_type                               |sector_id|sector                       |record_count|total_generation_mwh|
+------------+----------------------------------------+---------+-----------------------------+------------+--------------------+
|ALL         |all fuels                               |1        |Electric Utility             |72          |445803.6            |
|ALL         |all fuels                               |2        |IPP Non-CHP                  |72          |113582.77           |
|ALL         |all fuels                               |3        |IPP CHP                      |72          |8765.55             |
|ALL         |all fuels                               |4        |Commercial Non-CHP           |72          |3712.55             |
|ALL         |all fuels                               |5        |Commercial CHP           

In [0]:
from pyspark.sql.functions import col, sum, when

gold_df = (
    spark.table("virginia_energy_silver")
    .filter(col("sector_id") == "99")
    .groupBy("period_date")
    .agg(
        sum(when(col("fuel_type_id") == "ALL", col("generation_mwh"))).alias("total_generation_mwh"),
        sum(when(col("fuel_type_id") == "NG", col("generation_mwh"))).alias("natural_gas_mwh"),
        sum(when(col("fuel_type_id") == "NUC", col("generation_mwh"))).alias("nuclear_mwh"),
        sum(when(col("fuel_type_id") == "WND", col("generation_mwh"))).alias("wind_mwh"),
        sum(when(col("fuel_type_id") == "SUN", col("generation_mwh"))).alias("solar_mwh"),
        sum(when(col("fuel_type_id") == "HYC", col("generation_mwh"))).alias("hydro_mwh"),
        sum(when(col("fuel_type_id") == "BIO", col("generation_mwh"))).alias("biomass_mwh"),
        sum(when(col("fuel_type_id") == "COL", col("generation_mwh"))).alias("coal_mwh")
    )
    .orderBy("period_date")
)

display(gold_df)

period_date,total_generation_mwh,natural_gas_mwh,nuclear_mwh,wind_mwh,solar_mwh,hydro_mwh,biomass_mwh,coal_mwh
2020-01-01,9629.8083,6052.47955,2751.48,null,65.74472,199.91025,300.93506,259.03368
2020-02-01,8689.30008,5375.23449,2560.616,null,70.50936,223.12581,294.36155,185.98312
2020-03-01,9022.89881,5530.30259,2722.646,null,98.09204,250.91243,315.97153,108.96159
2020-04-01,7956.9851,4854.35286,2376.517,null,125.95828,216.29926,254.53082,98.60698
2020-05-01,7691.09107,5028.26586,2082.219,null,135.10082,218.26494,245.28409,3.3213
2020-06-01,8576.53187,5317.52164,2492.87,null,152.49186,158.90617,265.80445,237.72832
2020-07-01,10938.00183,6660.24018,2641.811,null,171.7923,141.63123,326.64022,1108.93431
2020-08-01,10066.33901,6257.61799,2650.968,null,122.32314,144.04705,305.13598,687.0608
2020-09-01,7691.99003,4896.41769,2160.84,null,108.91513,113.15597,303.1849,142.93906
2020-10-01,7007.29112,4234.86846,2294.293,null,115.69311,91.36641,248.29384,58.41411


In [0]:
from pyspark.sql.functions import col, sum, when, coalesce, lag, round, year, month
from pyspark.sql.window import Window

window_spec = Window.orderBy("period_date")
rolling_window = Window.orderBy("period_date").rowsBetween(-11, 0)

gold_df = (
    spark.table("virginia_energy_silver")
    .filter(col("sector_id") == "99")
    .groupBy("period_date")
    .agg(
        sum(when(col("fuel_type_id") == "ALL", col("generation_mwh"))).alias("total_generation_mwh"),
        sum(when(col("fuel_type_id") == "AOR", col("generation_mwh"))).alias("renewable_generation_mwh"),
        sum(when(col("fuel_type_id") == "NG", col("generation_mwh"))).alias("natural_gas_mwh"),
        sum(when(col("fuel_type_id") == "NUC", col("generation_mwh"))).alias("nuclear_mwh"),
        sum(when(col("fuel_type_id") == "WND", col("generation_mwh"))).alias("wind_mwh"),
        sum(when(col("fuel_type_id") == "SUN", col("generation_mwh"))).alias("solar_mwh"),
        sum(when(col("fuel_type_id") == "HYC", col("generation_mwh"))).alias("hydro_mwh"),
        sum(when(col("fuel_type_id") == "BIO", col("generation_mwh"))).alias("biomass_mwh"),
        sum(when(col("fuel_type_id") == "COL", col("generation_mwh"))).alias("coal_mwh")
    )
    .withColumn("year", year("period_date"))
    .withColumn("month", month("period_date"))
    .withColumn(
        "renewable_share_pct",
        round(
            col("renewable_generation_mwh") /
            col("total_generation_mwh") * 100,
            2
        )
    )
    .withColumn(
        "natural_gas_share_pct",
        round(
            col("natural_gas_mwh") /
            col("total_generation_mwh") * 100,
            2
        )
    )
    .withColumn(
        "nuclear_share_pct",
        round(
            col("nuclear_mwh") /
            col("total_generation_mwh") * 100,
            2
        )
    )
    .withColumn(
        "coal_share_pct",
        round(
            col("coal_mwh") /
            col("total_generation_mwh") * 100,
            2
        )
    )
    .withColumn(
        "prior_year_generation_mwh",
        lag("total_generation_mwh", 12).over(window_spec)
    )
    .withColumn(
        "yoy_generation_change_pct",
        round(
            (
                col("total_generation_mwh") -
                col("prior_year_generation_mwh")
            ) /
            col("prior_year_generation_mwh") * 100,
            2
        )
    )
    .withColumn(
        "rolling_12m_generation_mwh",
        round(
            sum("total_generation_mwh").over(rolling_window),
            2
        )
    )
    .orderBy("period_date")
)

display(gold_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


period_date,total_generation_mwh,renewable_generation_mwh,natural_gas_mwh,nuclear_mwh,wind_mwh,solar_mwh,hydro_mwh,biomass_mwh,coal_mwh,year,month,renewable_share_pct,natural_gas_share_pct,nuclear_share_pct,coal_share_pct,prior_year_generation_mwh,yoy_generation_change_pct,rolling_12m_generation_mwh
2020-01-01,9629.8083,366.67977,6052.47955,2751.48,null,65.74472,199.91025,300.93506,259.03368,2020,1,3.81,62.85,28.57,2.69,null,null,9629.81
2020-02-01,8689.30008,364.87091,5375.23449,2560.616,null,70.50936,223.12581,294.36155,185.98312,2020,2,4.2,61.86,29.47,2.14,null,null,18319.11
2020-03-01,9022.89881,414.06356,5530.30259,2722.646,null,98.09204,250.91243,315.97153,108.96159,2020,3,4.59,61.29,30.17,1.21,null,null,27342.01
2020-04-01,7956.9851,380.48911,4854.35286,2376.517,null,125.95828,216.29926,254.53082,98.60698,2020,4,4.78,61.01,29.87,1.24,null,null,35298.99
2020-05-01,7691.09107,380.38491,5028.26586,2082.219,null,135.10082,218.26494,245.28409,3.3213,2020,5,4.95,65.38,27.07,0.04,null,null,42990.08
2020-06-01,8576.53187,418.29631,5317.52164,2492.87,null,152.49186,158.90617,265.80445,237.72832,2020,6,4.88,62.0,29.07,2.77,null,null,51566.62
2020-07-01,10938.00183,498.43251,6660.24018,2641.811,null,171.7923,141.63123,326.64022,1108.93431,2020,7,4.56,60.89,24.15,10.14,null,null,62504.62
2020-08-01,10066.33901,427.45912,6257.61799,2650.968,null,122.32314,144.04705,305.13598,687.0608,2020,8,4.25,62.16,26.33,6.83,null,null,72570.96
2020-09-01,7691.99003,412.10003,4896.41769,2160.84,null,108.91513,113.15597,303.1849,142.93906,2020,9,5.36,63.66,28.09,1.86,null,null,80262.95
2020-10-01,7007.29112,363.98695,4234.86846,2294.293,null,115.69311,91.36641,248.29384,58.41411,2020,10,5.19,60.44,32.74,0.83,null,null,87270.24


In [0]:
gold_table = "virginia_energy_gold_generation_mix"

(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_table)
)

print("Gold table created successfully.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold table created successfully.


In [0]:
spark.sql("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(DISTINCT period_date) AS distinct_months,
        MIN(period_date) AS earliest_date,
        MAX(period_date) AS latest_date,
        ROUND(MIN(renewable_share_pct), 2) AS min_renewable_share_pct,
        ROUND(MAX(renewable_share_pct), 2) AS max_renewable_share_pct,
        ROUND(MIN(natural_gas_share_pct), 2) AS min_natural_gas_share_pct,
        ROUND(MAX(natural_gas_share_pct), 2) AS max_natural_gas_share_pct
    FROM virginia_energy_gold_generation_mix
""").show()

+-------------+---------------+-------------+-----------+-----------------------+-----------------------+-------------------------+-------------------------+
|total_records|distinct_months|earliest_date|latest_date|min_renewable_share_pct|max_renewable_share_pct|min_natural_gas_share_pct|max_natural_gas_share_pct|
+-------------+---------------+-------------+-----------+-----------------------+-----------------------+-------------------------+-------------------------+
|           72|             72|   2020-01-01| 2025-12-01|                   3.81|                  16.23|                    39.41|                    65.38|
+-------------+---------------+-------------+-----------+-----------------------+-----------------------+-------------------------+-------------------------+

